# 0. Data Acquisition: Harvesting AI Research Trends from arXiv

To quantify the pace of innovation in Artificial Intelligence, we programmatically collect publication metadata from **arXiv** using the **OAI-PMH (Open Archives Initiative Protocol for Metadata Harvesting)**. 

### Implementation Details:
* **Source:** arXiv.org via the `sickle` library.
* **Filtering:** We apply server-side filters for the computer science sub-category **(cs.AI)** to ensure thematic relevance.
* **Granularity:** Data is aggregated at a **daily level**, counting the number of new papers submitted each day.
* **Efficiency:** By leveraging server-side parameters (`from` and `until`), we minimize data transfer and focus specifically on our research timeframe (2016-2023).

The resulting dataset provides a daily "innovation count" which will be compared against financial market movements in the following sections.

In [1]:
from sickle import Sickle
from collections import defaultdict
import csv
from datetime import datetime

# Initialize OAI-PMH connection to arXiv
sickle = Sickle('https://oaipmh.arxiv.org/oai')

print("Fetching AI papers directly from arXiv API...")
print("Both category (cs:cs:AI) and date filters are applied server-side for maximum speed.")

# Applying BOTH category and date filters directly to the server!
api_params = {
    'metadataPrefix': 'arXiv',
    'set': 'cs:cs:AI',  # Fetching ONLY Artificial Intelligence papers directly
    'from': '2016-09-01',
    'until': '2024-01-01'
}

# Fetching records with our highly optimized parameters
records = sickle.ListRecords(**api_params)

daily_counts = defaultdict(int)

# Iterate through the fetched AI records and aggregate daily counts
for record in records:
    data = record.metadata
    
    # Skip the record if it lacks essential metadata
    if not data or "created" not in data:
        continue

    # Extract the creation date
    date_str = data["created"][0]
    
    try:
        # Convert string to date object
        paper_date = datetime.strptime(date_str, "%Y-%m-%d").date()
        daily_counts[str(paper_date)] += 1
    except ValueError:
        # Safely skip records with invalid date formats
        continue 

# Export the aggregated data to a CSV file
with open("ai_daily_papers.csv", "w", newline="") as f:    
    writer = csv.writer(f)
    writer.writerow(["date", "paper_count"])
    
    # Sort dates chronologically before writing
    for d in sorted(daily_counts):
        writer.writerow([d, daily_counts[d]])

print("Data successfully collected and saved to 'ai_daily_papers.csv'.")

Fetching AI papers directly from arXiv API...
Both category (cs:cs:AI) and date filters are applied server-side for maximum speed.
Data successfully collected and saved to 'ai_daily_papers.csv'.
